# Enhanced Image Generation with Real-World SwinIR (Replicate API)
This notebook uses the `requests` library to communicate cleanly with the Replicate API, bypassing dependency issues.
We target the official `jingyunliang/swinir` model with `Real-World Image Super-Resolution-Large` to specifically recover details in noisy, compressed imagery like vehicle snapshots.

In [1]:
# Dependencies are generally built-in or straightforward
%pip install -q requests

In [2]:
import os
import time
import base64
import requests
import shutil

# Read Replicate API Token robustly
try:
    with open('Replicate Access Token.txt', 'r') as f:
        REPLICATE_API_TOKEN = f.read().strip()
except FileNotFoundError:
    # Replace directly if the file isn't around
    REPLICATE_API_TOKEN = 'your_token_here'
    print("Warning: Token file not found. Set it manually.")

print("Loaded Replicate API Token successfully.")

def upscale_image_replicate(image_path):
    # Convert to Data URI
    with open(image_path, "rb") as f:
        data = f.read()
    b16 = base64.b64encode(data).decode('utf-8')
    mime = "image/jpeg" if image_path.lower().endswith('.jpg') else "image/png"
    data_uri = f"data:{mime};base64,{b16}"

    url = "https://api.replicate.com/v1/predictions"
    payload = {
        "version": "660d922d33153019e8c263a3bba265de882e7f4f70396546b6c9c8f9d47a021a",
        "input": {
            "image": data_uri,
            "task_type": "Real-World Image Super-Resolution-Large",
            "noise": 15,
            "jpeg": 40
        }
    }
    headers = {
        "Authorization": f"Bearer {REPLICATE_API_TOKEN}",
        "Content-Type": "application/json"
    }

    # Create Prediction
    resp = requests.post(url, json=payload, headers=headers)
    if resp.status_code == 402:
        raise Exception("402 Payment Required: Your Replicate account requires billing setup to use this API.")
    resp.raise_for_status()
    prediction = resp.json()

    # Poll for result
    get_url = prediction["urls"]["get"]
    while True:
        r = requests.get(get_url, headers=headers)
        r.raise_for_status()
        status = r.json()["status"]
        if status == "succeeded":
            return r.json()["output"] # Outputs list of strings (URI)
        elif status == "failed":
            raise Exception("Prediction failed: " + str(r.json().get("error")))
        time.sleep(1)


Loaded Replicate API Token successfully.


In [3]:
input_directory = os.path.join('Vehicle Snapshots', 'Bikes (Number-Plates On The Rear Mud-Guard)')
output_directory = os.path.join('Vehicle Snapshots', 'Enhanced Bike Snapshots (Replicate)')
os.makedirs(output_directory, exist_ok=True)

print(f"Scanning Directory: {input_directory}...")

if not os.path.isdir(input_directory):
    print(f"Directory {input_directory} not found. Please check the path.")
else:
    for root, dirs, files in os.walk(input_directory):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                file_path = os.path.join(root, file)
                new_filename = f"recreated_{file}"
                save_path = os.path.join(output_directory, new_filename)
                
                if os.path.exists(save_path):
                    print(f"Skipping {file_path}, enhanced image already exists.")
                    continue

                print(f"Loading {file_path}...")
                
                try:
                    max_retries = 3
                    success = False
                    
                    for attempt in range(max_retries):
                        try:
                            # Dispatch upscaling
                            output_urls = upscale_image_replicate(file_path)
                            output_url = output_urls[0] if isinstance(output_urls, list) else output_urls
                            
                            # Download actual image bytes into output_directory
                            img_data = requests.get(output_url).content
                            with open(save_path, 'wb') as handler:
                                handler.write(img_data)
                            
                            success = True
                            break
                                    
                        except Exception as e:
                            if "402" in str(e):
                                raise e
                            if attempt < max_retries - 1:
                                print(f"  -> Request error ({e}). Retrying attempt {attempt + 1} after 5 seconds...")
                                time.sleep(5)
                            else:
                                raise e
                    
                    if success and os.path.exists(save_path):
                        print(f"Successfully upscaled and saved image to: {save_path}")
                    else:
                        print(f"Failed to generate image data for {file}")
                    
                except Exception as e:
                    print(f"Failed to process file: {e}")
                    if "402" in str(e):
                        print("Stopping automated looping due to billing requirements.")
                        break

print("Finished scanning and upscaling files.")

Scanning Directory: Vehicle Snapshots\Bikes (Number-Plates On The Rear Mud-Guard)...
Loading Vehicle Snapshots\Bikes (Number-Plates On The Rear Mud-Guard)\T00217_bike.jpg...
Failed to process file: 402 Payment Required: Your Replicate account requires billing setup to use this API.
Stopping automated looping due to billing requirements.
Finished scanning and upscaling files.
